In [1]:
!pip uninstall -y numpy pyarrow pandas datasets transformers faiss faiss-cpu

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pyarrow 12.0.1
Uninstalling pyarrow-12.0.1:
  Successfully uninstalled pyarrow-12.0.1
Found existing installation: pandas 2.1.4
Uninstalling pandas-2.1.4:
  Successfully uninstalled pandas-2.1.4
Found existing installation: datasets 2.14.6
Uninstalling datasets-2.14.6:
  Successfully uninstalled datasets-2.14.6
Found existing installation: transformers 4.30.0
Uninstalling transformers-4.30.0:
  Successfully uninstalled transformers-4.30.0
Found existing installation: faiss-cpu 1.7.4
Uninstalling faiss-cpu-1.7.4:
  Successfully uninstalled faiss-cpu-1.7.4


In [2]:
!pip install -q \
transformers==4.30.0 \
datasets==2.14.6 \
faiss-cpu==1.7.4 \
numpy==1.26.4 \
pandas==2.1.4 \
pyarrow==12.0.1 \
sentencepiece \
accelerate \
evaluate \
sacrebleu \
scikit-learn \
tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grader-service 0.12.0 requires pandas>=2.2.3, but you have pandas 2.1.4 which is incompatible.
jax 0.9.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.9.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytesmo 0.18.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
obspy 1.4.2 requires sqlalchemy<2, but you have sqlalchemy 2.0.43 which is incompatible.
pygimli 1.5.4 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
tetgen 0.6.7 requires numpy<3,>=2, but you have numpy 1.26.4 which is incompatible.
xarray 2025.9.0 requires pandas>=2.2, but you have pandas 2.1.4 which is incompatible.
torchtext 0.18.0 requires torch>=2.3.0, but you have torch 2.2.2+cu121 which is incompatible.
opencv-python-headless 4.12.0.88 require

In [3]:
!pip install -q \
torch==2.2.2 \
torchvision==0.17.2 \
torchaudio==2.2.2 \
--index-url https://download.pytorch.org/whl/cu121

In [4]:
from datasets import load_dataset

import torch

from transformers import (

    RagTokenizer,
    RagRetriever,
    RagTokenForGeneration,

    Trainer,
    TrainingArguments,

    default_data_collator,
    set_seed
)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:",device)

/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violation

DEVICE: cuda


In [5]:
dataset = load_dataset(
    "search_qa",
    "train_test_val"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['category', 'air_date', 'question', 'value', 'answer', 'round', 'show_number', 'search_results'],
        num_rows: 151295
    })
    test: Dataset({
        features: ['category', 'air_date', 'question', 'value', 'answer', 'round', 'show_number', 'search_results'],
        num_rows: 43228
    })
    validation: Dataset({
        features: ['category', 'air_date', 'question', 'value', 'answer', 'round', 'show_number', 'search_results'],
        num_rows: 21613
    })
})


In [6]:
train_ds = dataset["train"].select(range(80000))
val_ds = dataset["validation"]
test_ds = dataset["test"]

print("TRAIN:",len(train_ds))
print("VAL:",len(val_ds))
print("TEST:",len(test_ds))

TRAIN: 80000
VAL: 21613
TEST: 43228


In [7]:
def preprocess(example):

    return {
        "source": str(example["answer"]),
        "target": str(example["question"])
    }

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

In [8]:
from transformers import (
    RagTokenizer,
    RagRetriever,
    RagTokenForGeneration
)

In [9]:
tokenizer_tok = RagTokenizer.from_pretrained(
    "facebook/rag-token-nq"
)

retriever_tok = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="compressed",
    use_dummy_dataset=False
)

rag_token = RagTokenForGeneration.from_pretrained(
    "facebook/rag-token-nq",
    retriever=retriever_tok
).to(device)

rag_token.config.n_docs = 10

print("RAG TOKEN MODEL READY")

/opt/micromamba/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/opt/micromamba/lib/python3.11/site-packages/transformers/models/bart/configuration_bart.py:179: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected

RAG TOKEN MODEL READY


In [10]:
def tokenize_tok(batch):

    model_inputs = tokenizer_tok.question_encoder(
        batch["source"],
        max_length=32,
        truncation=True,
        padding="max_length"
    )

    with tokenizer_tok.generator.as_target_tokenizer():

        labels = tokenizer_tok.generator(
            batch["target"],
            max_length=32,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [11]:
train_tok = train_ds.map(
    tokenize_tok,
    batched=True,
    remove_columns=train_ds.column_names
)

val_tok = val_ds.map(
    tokenize_tok,
    batched=True,
    remove_columns=val_ds.column_names
)

test_tok = test_ds.map(
    tokenize_tok,
    batched=True,
    remove_columns=test_ds.column_names
)

Map:   0%|          | 0/43228 [00:00<?, ? examples/s]

/opt/micromamba/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:3619: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [12]:
from transformers import TrainingArguments

In [13]:
for param in rag_token.question_encoder.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir="./rag_token",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    evaluation_strategy="no",
    save_steps=1000,
    logging_steps=100,
    save_total_limit=2,
    report_to="none"
)

In [14]:
from transformers import Trainer

In [15]:
class RagTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False
    ):

        outputs = model(**inputs)

        loss = outputs.loss.mean()

        return (
            (loss,outputs)
            if return_outputs
            else loss
        )

In [16]:
trainer = RagTrainer(
    model=rag_token,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer_tok.generator,
    data_collator=default_data_collator,
)

In [17]:
trainer.train()

/opt/micromamba/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss
100,182.473100
200,99.888900
300,82.875400
400,78.490700
500,76.267500
600,74.837400
700,74.553300
800,71.441300
900,70.431100
1000,70.640100


TrainOutput(global_step=10000, training_loss=64.76223286132813, metrics={'train_runtime': 50972.4972, 'train_samples_per_second': 3.139, 'train_steps_per_second': 0.196, 'total_flos': 1.344844333056e+16, 'train_loss': 64.76223286132813, 'epoch': 2.0})

In [24]:
# !pip install -q sacrebleu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [20]:
references = []
predictions = []

batch = val_ds[:10000]

for source, target in zip(batch["source"], batch["target"]):
    
    inputs = tokenizer_tok.question_encoder(
        source,
        return_tensors="pt",
        truncation=True,
        max_length=32
    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
        if k!="token_type_ids"
    }

    with torch.no_grad():

        outputs = rag_token.generate(
            **inputs,
            num_beams=4,
            max_new_tokens=50
        )

    pred = tokenizer_tok.generator.decode(
        outputs[0],
        skip_special_tokens=True
    )

    references.append(target)
    predictions.append(pred)


with open("references.txt","w") as f:
    for q in references:
        f.write(q.strip()+"\n")

with open("hypotheses.txt","w") as f:
    for q in predictions:
        f.write(q.strip()+"\n")

In [21]:
import subprocess
import re

cmd = [
    "python",
    "Answerability-Metric/answerability_score.py",
    "--data_type", "squad",
    "--ref_file", "references.txt",
    "--hyp_file", "hypotheses.txt",
    "--ner_weight", "0.41",
    "--qt_weight", "0.20",
    "--re_weight", "0.36",
    "--delta", "0.66",
    "--ngram_metric", "Bleu_1"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

# Extract the official scores

text = result.stderr

qbleu1 = float(
    re.search(
        r"Mean Answerability Score Across Questions:\s*([\d.]+)",
        result.stderr
    ).group(1)
)

bleu1 = float(
    re.search(
        r"N-gram Score:\s*([\d.]+)",
        result.stderr
    ).group(1)
)

print(f"Official Q-BLEU-1: {100*qbleu1:.2f}")
print(f"Official BLEU-1: {100*bleu1:.2f}")


Official Q-BLEU-1: 19.10
Official BLEU-1: 14.20


In [28]:
trainer.save_model("./rag_token_final")

In [22]:
examples = [
    "Albert Einstein",
    "The Beatles",
    "Leonardo da Vinci",
    "Michael Jordan",
    "Marie Curie",
    "William Shakespeare",
    "Alexander The Great"
]

rag_token.eval()

for entity in examples:

    inputs = tokenizer_tok.question_encoder(
        entity,
        return_tensors="pt"

    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
        if k!="token_type_ids"
    }

    with torch.no_grad():

        outputs = rag_token.generate(
            **inputs,
            num_beams=4,
            max_new_tokens=50,
            early_stopping=True
        )

    prediction = tokenizer_tok.generator.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    print("\n"+"="*100)
    print("ENTITY:")
    print(entity)
    print("\nGENERATED JEOPARDY CLUE:")
    print(prediction)

/opt/micromamba/lib/python3.11/site-packages/transformers/generation/utils.py:2838: UserWarning: `max_length` is deprecated in this function, use `stopping_criteria=StoppingCriteriaList(MaxLengthCriteria(max_length=max_length))` instead.
  warnings.warn(



ENTITY:
Albert Einstein

GENERATED JEOPARDY CLUE:
This German-Swedish physicist's theory of general quantum states is known as the "mirror theory"

ENTITY:
The Beatles

GENERATED JEOPARDY CLUE:
This group's first No. 1 hit was "Love Me Tender"

ENTITY:
Leonardo da Vinci

GENERATED JEOPARDY CLUE:
In 1519 this Renaissance artist died at the age of 67 & his body was buried in the chapel of the Medici family in Florence

ENTITY:
Michael Jordan

GENERATED JEOPARDY CLUE:
He's the only player in NBA history to lead the league in points, rebounds & steals in a single season

ENTITY:
Marie Curie

GENERATED JEOPARDY CLUE:
She was born in Paris in 1887, the daughter of a doctor & a chemist

ENTITY:
William Shakespeare

GENERATED JEOPARDY CLUE:
In 1616 this playwright died in Stratford-upon-Avon, War & Peace

ENTITY:
Alexander The Great

GENERATED JEOPARDY CLUE:
In 334 B.C. this Macedonian conqueror defeated the Persians at the Battle of Granicus
